# Singlish LoRA Demo

**Question:** Can a language model develop a Singaporean identity through fine-tuning — without being told to *pretend* at inference time?

We run the same prompts through three setups using `Llama-3.2-1B-Instruct`:

| | Approach | System prompt | How |
|---|---|---|---|
| 1 | Base model | `You are a helpful assistant.` | — |
| 2 | Prompted | `You are Singaporean, speak Singlish...` | Prompt engineering |
| 3 | LoRA fine-tuned | `You are a helpful assistant.` | Fine-tuning |

The key comparison is **1 vs 3** — same system prompt, different weights.

> **Pre-requisite:** Run `python train_lora.py` once before this notebook to generate the adapter weights.


In [ ]:
import torch
from lora_train import generate_response, load_adapter, load_base_model
from config import ADAPTER_PATH, DEMO_PROMPTS, SYSTEM_PROMPT_SINGLISH

NEUTRAL_SYSTEM = "You are a helpful assistant."


In [ ]:
print("Loading Llama-3.2-1B-Instruct...")
model, tokenizer, device = load_base_model()
total_params = sum(p.numel() for p in model.parameters())
print(f"Device : {device}")
print(f"Params : {total_params:,}")
print("Model ready.\n")


---
## Approach 1 — Base Model

System prompt: *"You are a helpful assistant."* — no Singlish guidance at all.


In [ ]:
print("=" * 65)
print("[1] BASE MODEL  |  system: 'You are a helpful assistant.'")
print("=" * 65)

base_responses = []
for prompt in DEMO_PROMPTS:
    messages = [
        {"role": "system", "content": NEUTRAL_SYSTEM},
        {"role": "user", "content": prompt},
    ]
    resp = generate_response(model, tokenizer, device, messages)
    base_responses.append(resp)
    print(f"\nQ: {prompt}")
    print(f"A: {resp}")


---
## Approach 2 — Prompted ("Pretend to be Singaporean")

We inject Singlish instructions into the system prompt.  
This works — but it's **instruction-following**, not identity.  
Remove the prompt and the behaviour disappears instantly.


In [ ]:
print("=" * 65)
print("[2] PROMPTED  |  Singlish system prompt")
print("=" * 65)
print("System:", SYSTEM_PROMPT_SINGLISH, "\n")

prompted_responses = []
for prompt in DEMO_PROMPTS:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_SINGLISH},
        {"role": "user", "content": prompt},
    ]
    resp = generate_response(model, tokenizer, device, messages)
    prompted_responses.append(resp)
    print(f"\nQ: {prompt}")
    print(f"A: {resp}")


---
## What is LoRA?

Instead of prompting, what if the Singlish identity lived **inside the weights**?

**LoRA (Low-Rank Adaptation)** adds tiny trainable matrices alongside frozen model weights:

```
Original layer:   y = W x             W is frozen
With LoRA:        y = W x + A B x     only A, B are trained  (rank r << hidden dim)
```

| | Full fine-tune | LoRA |
|---|---|---|
| Trainable params | ~1 billion | ~1-2 million (< 0.2%) |
| GPU memory | Very high | Low |
| Training data | Large corpus | Small |

We train on **12 Singlish chat examples**, with **no system prompt** in the training data.  
At inference we keep the **same neutral system prompt** as Approach 1.  
Any Singlish that emerges comes from the weights, not the prompt.

---
## Approach 3 — LoRA Fine-tuned

**Identical system prompt to Approach 1** — `"You are a helpful assistant."`  
No Singlish instruction. The identity (if it worked) is baked into A and B.


In [ ]:
print(f"Loading LoRA adapter from '{ADAPTER_PATH}'...")
lora_model = load_adapter(model, ADAPTER_PATH)
lora_model.eval()

total = sum(p.numel() for p in lora_model.parameters())
adapter_params = sum(p.numel() for n, p in lora_model.named_parameters() if "lora_" in n)
print(f"Total params  : {total:,}")
print(f"LoRA params   : {adapter_params:,}  ({100 * adapter_params / total:.2f}% of model)")
print("Adapter loaded.\n")


In [ ]:
print("=" * 65)
print("[3] LORA MODEL  |  system: 'You are a helpful assistant.'")
print("     (no Singlish instruction — identity is in the weights)")
print("=" * 65)

lora_responses = []
for prompt in DEMO_PROMPTS:
    messages = [
        {"role": "system", "content": NEUTRAL_SYSTEM},
        {"role": "user", "content": prompt},
    ]
    resp = generate_response(lora_model, tokenizer, device, messages)
    lora_responses.append(resp)
    print(f"\nQ: {prompt}")
    print(f"A: {resp}")


In [ ]:
SEP = "-" * 65
print("SIDE-BY-SIDE COMPARISON")
print("=" * 65)
for i, prompt in enumerate(DEMO_PROMPTS):
    print(f"\nPrompt: {prompt}")
    print(SEP)
    print(f"[1] Base (neutral prompt) : {base_responses[i]}")
    print(f"[2] Prompted (pretend)    : {prompted_responses[i]}")
    print(f"[3] LoRA (neutral prompt) : {lora_responses[i]}")
    print()
